In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt

from scipy.special import ellipk, ellipj

from ipywidgets import (
    FloatSlider,
    HTML,
    HTMLMath,
    VBox,
    HBox,
    Layout
)

from IPython.display import display

# ============================================================
# JACOBI ELLIPTIC FUNCTIONS
#
# sn(u,k), cn(u,k), dn(u,k)
#
# IMPORTANT:
# scipy.special uses the parameter m = k^2
# ============================================================

plt.ioff()

# ============================================================
# JUPYTER / BINDER DISPLAY SETTINGS
# ============================================================

display(HTML("""
<style>

.container {
    width: 98% !important;
    max-width: none !important;
}

.output_area,
.output_subarea {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.output_scroll {
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
    box-shadow: none !important;
}

.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow: visible !important;
    resize: none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display: none !important;
}

.jac-title {
    font-family: Arial, sans-serif;
    font-size: 20px;
    font-weight: bold;
    color: #6f3fa0;
}

.jac-label {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
}

.jac-value {
    font-family: Arial, sans-serif;
    font-size: 14px;
    font-weight: bold;
    color: #0b3d91;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1200px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="jac-title" style="margin-bottom:8px;">
Jacobi Elliptic Functions
</div>

<div style="margin-bottom:5px;">
The Jacobi elliptic functions sn(u,k), cn(u,k) and dn(u,k)
are obtained from the inversion of the incomplete elliptic
integral of the first kind.
</div>

<div style="margin-bottom:5px;">
The horizontal axis is normalized as u/K(k). Therefore, the
real period 4K(k) always corresponds to a normalized interval
of length 4. Two complete real periods are displayed.
</div>

<div>
<b>This notebook:</b> shows how the Jacobi elliptic functions
change with the elliptic modulus k and verifies their fundamental
identities numerically at a selected value of u/K.
</div>

</div>
""")

# ============================================================
# MATHEMATICAL DEFINITIONS
# ============================================================

definition_sn = HTMLMath(
    value=(
        r'\('
        r'\operatorname{sn}(u,k)'
        r'='
        r'\sin[\operatorname{am}(u,k)]'
        r'\)'
    )
)

definition_cn = HTMLMath(
    value=(
        r'\('
        r'\operatorname{cn}(u,k)'
        r'='
        r'\cos[\operatorname{am}(u,k)]'
        r'\)'
    )
)

definition_dn = HTMLMath(
    value=(
        r'\('
        r'\operatorname{dn}(u,k)'
        r'='
        r'\sqrt{1-k^2\operatorname{sn}^2(u,k)}'
        r'\)'
    )
)

definitions_row = HBox(
    [
        definition_sn,
        definition_cn,
        definition_dn
    ],
    layout=Layout(
        width='1180px',
        gap='25px',
        align_items='center',
        overflow='visible'
    )
)

# ============================================================
# SLIDER STYLES
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='250px'
)

label_layout = Layout(
    width='155px',
    min_width='155px'
)

value_layout = Layout(
    width='65px',
    min_width='65px',
    margin='0px 0px 0px 6px'
)

row_layout = Layout(
    width='490px',
    height='40px',
    align_items='center'
)

# ============================================================
# ELLIPTIC MODULUS k
# ============================================================

k_slider = FloatSlider(
    min=0.00,
    max=0.99,
    step=0.01,
    value=0.70,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

k_label = HTML(
    '<div class="jac-label">Elliptic modulus k:</div>',
    layout=label_layout
)

k_value = HTML(
    '<div class="jac-value">0.70</div>',
    layout=value_layout
)

k_row = HBox(
    [
        k_label,
        k_slider,
        k_value
    ],
    layout=row_layout
)

# ============================================================
# NORMALIZED POSITION v = u/K
# ============================================================

v_slider = FloatSlider(
    min=-4.0,
    max=4.0,
    step=0.02,
    value=1.0,
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

v_label = HTML(
    '<div class="jac-label">Normalized u/K:</div>',
    layout=label_layout
)

v_value = HTML(
    '<div class="jac-value">1.00</div>',
    layout=value_layout
)

v_row = HBox(
    [
        v_label,
        v_slider,
        v_value
    ],
    layout=row_layout
)

# ============================================================
# PARAMETER PANEL
# ============================================================

parameters_panel = VBox(
    [
        HTML("""
        <div class="jac-title" style="margin-bottom:8px;">
            Parameters
        </div>
        """),

        k_row,
        v_row
    ],
    layout=Layout(
        width='520px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# CURRENT VALUES
# ============================================================

K_math = HTMLMath()
u_math = HTMLMath()

sn_math = HTMLMath()
cn_math = HTMLMath()
dn_math = HTMLMath()

K_math.layout = Layout(
    width='210px'
)

u_math.layout = Layout(
    width='210px'
)

sn_math.layout = Layout(
    width='190px'
)

cn_math.layout = Layout(
    width='190px'
)

dn_math.layout = Layout(
    width='190px'
)

current_values_panel = VBox(
    [
        HTML("""
        <div class="jac-title" style="margin-bottom:8px;">
            Current Values
        </div>
        """),

        HBox(
            [
                K_math,
                u_math
            ],
            layout=Layout(
                width='470px',
                gap='15px',
                overflow='visible'
            )
        ),

        HBox(
            [
                sn_math,
                cn_math,
                dn_math
            ],
            layout=Layout(
                width='610px',
                gap='12px',
                overflow='visible'
            )
        )
    ],
    layout=Layout(
        width='650px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# TOP ROW
# ============================================================

top_row = HBox(
    [
        parameters_panel,
        current_values_panel
    ],
    layout=Layout(
        width='1190px',
        gap='15px',
        align_items='stretch',
        overflow='visible'
    )
)

# ============================================================
# NORMALIZED HORIZONTAL AXIS
#
# v = u/K
#
# -4 <= v <= 4 gives two real periods because
# the period of sn and cn is 4K.
# ============================================================

v_axis = np.linspace(
    -4.0,
    4.0,
    1800
)

# ============================================================
# INITIAL DATA
# ============================================================

k0 = k_slider.value
m0 = k0**2

K0 = ellipk(
    m0
)

u_axis0 = (
    v_axis * K0
)

sn0, cn0, dn0, ph0 = ellipj(
    u_axis0,
    m0
)

# ============================================================
# MAIN FIGURE
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.0, 5.0)
)

fig.canvas.header_visible = False
fig.canvas.footer_visible = False
fig.canvas.toolbar_visible = False

fig.canvas.layout = Layout(
    width='800px',
    height='500px',
    overflow='visible'
)

ax.set_title(
    'Jacobi Elliptic Functions',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax.set_xlabel(
    'Normalized argument u / K',
    fontsize=11
)

ax.set_ylabel(
    'Function value',
    fontsize=11
)

ax.set_xlim(
    -4.0,
    4.0
)

ax.set_ylim(
    -1.15,
    1.15
)

ax.axhline(
    0.0,
    color='black',
    linewidth=1.0
)

ax.grid(
    True,
    linestyle=':',
    alpha=0.40
)

# ============================================================
# CURVES CREATED ONCE
# ============================================================

line_sn, = ax.plot(
    v_axis,
    sn0,
    linewidth=2.0,
    label='sn(u,k)'
)

line_cn, = ax.plot(
    v_axis,
    cn0,
    linewidth=2.0,
    label='cn(u,k)'
)

line_dn, = ax.plot(
    v_axis,
    dn0,
    linewidth=2.0,
    label='dn(u,k)'
)

# ============================================================
# CURRENT POSITION
# ============================================================

position_line = ax.axvline(
    v_slider.value,
    linestyle='--',
    linewidth=1.2,
    color='black',
    alpha=0.70
)

point_sn, = ax.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

point_cn, = ax.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

point_dn, = ax.plot(
    [],
    [],
    linestyle='None',
    marker='o',
    markersize=7
)

# ============================================================
# LEGEND BELOW FIGURE
# ============================================================

ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.15),
    ncol=3,
    fontsize=9,
    frameon=True
)

fig.subplots_adjust(
    left=0.10,
    right=0.97,
    top=0.90,
    bottom=0.23
)

# ============================================================
# FUNDAMENTAL IDENTITIES
# ============================================================

identity1_math = HTMLMath(
    value=(
        r'\('
        r'\operatorname{sn}^2(u,k)'
        r'+'
        r'\operatorname{cn}^2(u,k)'
        r'=1'
        r'\)'
    )
)

identity2_math = HTMLMath(
    value=(
        r'\('
        r'k^2\operatorname{sn}^2(u,k)'
        r'+'
        r'\operatorname{dn}^2(u,k)'
        r'=1'
        r'\)'
    )
)

identity1_value = HTMLMath()
identity2_value = HTMLMath()

identity_panel = VBox(
    [
        HTML("""
        <div class="jac-title" style="margin-bottom:8px;">
            Fundamental Identities
        </div>
        """),

        identity1_math,
        identity1_value,

        identity2_math,
        identity2_value
    ],
    layout=Layout(
        width='365px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# LIMITING CASES
# ============================================================

limiting_panel = HTML("""
<div style="
    width:335px;
    padding:9px 12px;
    border:1px solid #d7c7e5;
    font-family:Arial, sans-serif;
    font-size:14px;
    line-height:1.55;
    box-sizing:border-box;
    margin-top:10px;
">

<div style="
    color:#6f3fa0;
    font-size:17px;
    font-weight:bold;
    margin-bottom:6px;
">
Limiting Cases
</div>

<div>
<b>k = 0</b>
</div>

<div>
sn(u,0) = sin(u)
</div>

<div>
cn(u,0) = cos(u)
</div>

<div>
dn(u,0) = 1
</div>

<div style="margin-top:8px;">
<b>k → 1</b>
</div>

<div>
sn(u,k) → tanh(u)
</div>

<div>
cn(u,k) → sech(u)
</div>

<div>
dn(u,k) → sech(u)
</div>

</div>
""")

right_column = VBox(
    [
        identity_panel,
        limiting_panel
    ],
    layout=Layout(
        width='380px',
        overflow='visible'
    )
)

# ============================================================
# VISUAL ROW
# ============================================================

visual_row = HBox(
    [
        fig.canvas,
        right_column
    ],
    layout=Layout(
        width='1200px',
        gap='15px',
        align_items='flex-start',
        overflow='visible'
    )
)

# ============================================================
# REAL PERIOD INFORMATION
#
# NO \quad
# NO \qquad
# ============================================================

period_sn = HTMLMath(
    value=(
        r'\('
        r'\operatorname{sn}(u+4K,k)'
        r'='
        r'\operatorname{sn}(u,k)'
        r'\)'
    )
)

period_cn = HTMLMath(
    value=(
        r'\('
        r'\operatorname{cn}(u+4K,k)'
        r'='
        r'\operatorname{cn}(u,k)'
        r'\)'
    )
)

period_dn = HTMLMath(
    value=(
        r'\('
        r'\operatorname{dn}(u+2K,k)'
        r'='
        r'\operatorname{dn}(u,k)'
        r'\)'
    )
)

period_panel = HBox(
    [
        period_sn,
        period_cn,
        period_dn
    ],
    layout=Layout(
        width='1150px',
        gap='30px',
        padding='8px 12px',
        border='1px solid #d7c7e5',
        overflow='visible'
    )
)

# ============================================================
# UPDATE FUNCTION
#
# No clear_output()
# No figure recreation
# No axis rescaling
#
# Only existing curves and markers are updated.
# ============================================================

def update_notebook(change=None):

    k = (
        k_slider.value
    )

    v_current = (
        v_slider.value
    )

    # --------------------------------------------------------
    # SciPy parameter
    # --------------------------------------------------------

    m = (
        k**2
    )

    # --------------------------------------------------------
    # Complete elliptic integral K
    # --------------------------------------------------------

    K = ellipk(
        m
    )

    # --------------------------------------------------------
    # Entire curves
    # --------------------------------------------------------

    u_axis = (
        v_axis * K
    )

    (
        sn_values,
        cn_values,
        dn_values,
        ph_values
    ) = ellipj(
        u_axis,
        m
    )

    # --------------------------------------------------------
    # Update curves only
    # --------------------------------------------------------

    line_sn.set_ydata(
        sn_values
    )

    line_cn.set_ydata(
        cn_values
    )

    line_dn.set_ydata(
        dn_values
    )

    # --------------------------------------------------------
    # Selected point
    # --------------------------------------------------------

    u_current = (
        v_current * K
    )

    (
        sn_current,
        cn_current,
        dn_current,
        ph_current
    ) = ellipj(
        u_current,
        m
    )

    position_line.set_xdata(
        [
            v_current,
            v_current
        ]
    )

    point_sn.set_data(
        [v_current],
        [sn_current]
    )

    point_cn.set_data(
        [v_current],
        [cn_current]
    )

    point_dn.set_data(
        [v_current],
        [dn_current]
    )

    # --------------------------------------------------------
    # Slider readouts
    # --------------------------------------------------------

    k_value.value = (
        f'<div class="jac-value">{k:.2f}</div>'
    )

    v_value.value = (
        f'<div class="jac-value">{v_current:.2f}</div>'
    )

    # --------------------------------------------------------
    # Current numerical values
    #
    # Separate widgets are used.
    # No LaTeX spacing commands are required.
    # --------------------------------------------------------

    K_math.value = (
        r'\('
        r'K(k)='
        +
        f'{K:.6f}'
        +
        r'\)'
    )

    u_math.value = (
        r'\('
        r'u='
        +
        f'{u_current:.6f}'
        +
        r'\)'
    )

    sn_math.value = (
        r'\('
        r'\operatorname{sn}(u,k)='
        +
        f'{sn_current:.6f}'
        +
        r'\)'
    )

    cn_math.value = (
        r'\('
        r'\operatorname{cn}(u,k)='
        +
        f'{cn_current:.6f}'
        +
        r'\)'
    )

    dn_math.value = (
        r'\('
        r'\operatorname{dn}(u,k)='
        +
        f'{dn_current:.6f}'
        +
        r'\)'
    )

    # --------------------------------------------------------
    # Verify identities numerically
    # --------------------------------------------------------

    identity1 = (
        sn_current**2
        +
        cn_current**2
    )

    identity2 = (
        k**2
        *
        sn_current**2
        +
        dn_current**2
    )

    identity1_value.value = (
        r'\('
        r'\mathrm{Numerical\ value}='
        +
        f'{identity1:.12f}'
        +
        r'\)'
    )

    identity2_value.value = (
        r'\('
        r'\mathrm{Numerical\ value}='
        +
        f'{identity2:.12f}'
        +
        r'\)'
    )

    # --------------------------------------------------------
    # Redraw only
    # --------------------------------------------------------

    fig.canvas.draw_idle()

# ============================================================
# CONNECT CONTROLS
# ============================================================

k_slider.observe(
    update_notebook,
    names='value'
)

v_slider.observe(
    update_notebook,
    names='value'
)

# ============================================================
# INITIAL UPDATE
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        definitions_row,
        top_row,
        visual_row,
        period_panel
    ],
    layout=Layout(
        width='1220px',
        gap='10px',
        overflow='visible'
    )
)

display(
    main_layout
)